In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sbn
from warnings import filterwarnings

from mlxtend.frequent_patterns import apriori, association_rules
from prefixspan import PrefixSpan

filterwarnings("ignore")

In [10]:
mypivot = pd.read_csv("../../data/processed/MYpivot.csv")
tdmypivot = pd.read_csv("../../data/processed/TDMYpivot.csv")

# Apriori Analysis 1

Apriori ile zamandan bağımsız birliktelik analizleri

In [26]:
appri_df = pd.DataFrame(mypivot.values, columns=mypivot.columns)

frequent_itemsets = apriori(appri_df, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.4)

In [28]:
def recommend_oblasts(oblast):
    recommended_oblast = set()

    for o in oblast:
        related_rules = rules[rules['antecedents'] == {o}]

        for _, row in related_rules.iterrows():
            recommended_oblast.update(row['consequents'])

    return recommended_oblast

basket = ['donetsk','luhansk']

recommended_oblasts = recommend_oblasts(basket)
print("input oblast:", basket)
print("output oblast:", list(recommended_oblasts))

input oblast: ['donetsk', 'luhansk']
output oblast: ['kharkiv', 'luhansk', 'zaporizhzhia', 'kherson', 'donetsk', 'dnipropetrovsk', 'zaporizhia', 'belgorod']


In [19]:
print("Frekans Analizi")
frequent_itemsets.sort_values(by="support",ascending=False)

Frekans Analizi


,support,itemsets
425,1.000,"(luhansk, donetsk, kherson)"
69,1.000,"(donetsk, kherson)"
114,1.000,"(luhansk, kherson)"
3,1.000,(donetsk)
7,1.000,(kherson)
...,...,...
2827,0.125,"(kharkiv, zaporizhzhia, kherson, dnipropetrovs..."
2826,0.125,"(kharkiv, zaporizhzhia, kherson, dnipropetrovs..."
2825,0.125,"(kharkiv, kherson, dnipropetrovsk, zaporizhia,..."
2822,0.125,"(luhansk, kharkiv, tambov, kherson, dnipropetr..."


In [20]:
print("İlişki Kurallari")
rules

İlişki Kurallari


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(bryansk),(belgorod),0.375,0.875,0.375,1.000000,1.142857,0.046875,inf,0.200000
1,(belgorod),(bryansk),0.875,0.375,0.375,0.428571,1.142857,0.046875,1.09375,1.000000
2,(dnipropetrovsk),(belgorod),0.500,0.875,0.375,0.750000,0.857143,-0.062500,0.50000,-0.250000
3,(belgorod),(dnipropetrovsk),0.875,0.500,0.375,0.428571,0.857143,-0.062500,0.87500,-0.571429
4,(belgorod),(donetsk),0.875,1.000,0.875,1.000000,1.000000,0.000000,inf,0.000000
...,...,...,...,...,...,...,...,...,...,...
652662,(tver),"(kharkiv, luhansk, zaporizhzhia, novgorod, khe...",0.125,0.125,0.125,1.000000,8.000000,0.109375,inf,1.000000
652663,(novgorod),"(kharkiv, luhansk, tver, zaporizhzhia, kherson...",0.125,0.125,0.125,1.000000,8.000000,0.109375,inf,1.000000
652664,(samara),"(kharkiv, luhansk, tver, zaporizhzhia, novgoro...",0.125,0.125,0.125,1.000000,8.000000,0.109375,inf,1.000000
652665,(mykolaiv),"(kharkiv, luhansk, tver, zaporizhzhia, novgoro...",0.250,0.125,0.125,0.500000,4.000000,0.093750,1.75000,1.000000


# Apriori Analysis 2

In [29]:
appri_df = pd.DataFrame(tdmypivot.values, columns=tdmypivot.columns)

frequent_itemsets = apriori(appri_df, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.4)

In [30]:
basket = ['donetsk','luhansk']

recommended_oblasts = recommend_oblasts(basket)
print("input oblast:", basket)
print("output oblast:", list(recommended_oblasts))

input oblast: ['donetsk', 'luhansk']
output oblast: ['donetsk']


In [31]:
print("Frekans Analizi")
frequent_itemsets.sort_values(by="support",ascending=False)

Frekans Analizi


,support,itemsets
3,0.645570,(donetsk)
6,0.268987,(luhansk)
5,0.202532,(kherson)
8,0.189873,(zaporizhzhia)
4,0.164557,(kharkiv)
13,0.155063,"(luhansk, donetsk)"
12,0.104430,"(donetsk, kherson)"
11,0.075949,"(kharkiv, donetsk)"
15,0.072785,"(zaporizhzhia, donetsk)"
7,0.056962,(zaporizhia)


In [24]:
print("İlişki Kurallari")
rules

İlişki Kurallari


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(belgorod),(donetsk),0.047468,0.64557,0.028481,0.600000,0.929412,-0.002163,0.886076,-0.073846
1,(kharkiv),(donetsk),0.164557,0.64557,0.075949,0.461538,0.714932,-0.030284,0.658228,-0.323077
2,(kherson),(donetsk),0.202532,0.64557,0.104430,0.515625,0.798713,-0.026318,0.731727,-0.240132
3,(luhansk),(donetsk),0.268987,0.64557,0.155063,0.576471,0.892964,-0.018587,0.836850,-0.140873
4,"(kharkiv, kherson)",(donetsk),0.028481,0.64557,0.012658,0.444444,0.688453,-0.005728,0.637975,-0.317778
5,"(luhansk, kherson)",(donetsk),0.044304,0.64557,0.025316,0.571429,0.885154,-0.003285,0.827004,-0.119534


# PrefixSpan Analysis 1

In [33]:
time_series_data = mypivot.values

ps = PrefixSpan(time_series_data)
frequent_patterns = ps.frequent(2)  # En az 2 kez tekrar eden desenleri bul


In [34]:
prefixspan_results = frequent_patterns
oblast_names = mypivot.columns

# Desenleri oblast isimleri ile eşleştirme
matched_patterns = [(count, [oblast_names[i] for i, val in enumerate(pattern) if val]) for count, pattern in prefixspan_results]

df_matched_patterns = pd.DataFrame(matched_patterns, columns=['Count', 'Matched_Oblasts'])
df_matched_patterns


,Count,Matched_Oblasts
0,8,[belgorod]
1,8,"[belgorod, bryansk]"
2,8,"[belgorod, bryansk]"
3,8,"[belgorod, bryansk, donetsk]"
4,8,"[belgorod, bryansk, donetsk]"
...,...,...
13521,3,"[dnipropetrovsk, kaliningrad, kherson]"
13522,2,"[dnipropetrovsk, kaliningrad, kherson, luhansk]"
13523,2,"[dnipropetrovsk, kaliningrad, kherson]"
13524,2,"[dnipropetrovsk, kaliningrad, kherson]"


In [35]:
df_matched_patterns["Matched_Oblasts"].value_counts()

Matched_Oblasts
[belgorod]                                                15
[belgorod, bryansk]                                       15
[belgorod, dnipropetrovsk, donetsk]                       14
[bryansk]                                                 14
[]                                                        14
                                                          ..
[belgorod, dnipropetrovsk, kharkiv, kherson, samara]       1
[belgorod, dnipropetrovsk, kharkiv, kherson, smolensk]     1
[belgorod, dnipropetrovsk, kharkiv, kherson, sumy]         1
[belgorod, dnipropetrovsk, kharkiv, kherson, novgorod]     1
[dnipropetrovsk, kaliningrad, kherson, luhansk]            1
Name: count, Length: 8926, dtype: int64

# PrefixSpan Analysis 2

In [36]:
time_series_data = tdmypivot.values

ps = PrefixSpan(time_series_data)
frequent_patterns = ps.frequent(2)  # En az 2 kez tekrar eden desenleri bul


In [37]:
prefixspan_results = frequent_patterns
oblast_names = mypivot.columns

# Desenleri oblast isimleri ile eşleştirme
matched_patterns = [(count, [oblast_names[i] for i, val in enumerate(pattern) if val]) for count, pattern in prefixspan_results]

df_matched_patterns = pd.DataFrame(matched_patterns, columns=['Count', 'Matched_Oblasts'])
df_matched_patterns

,Count,Matched_Oblasts
0,316,[]
1,316,[]
2,316,[]
3,304,[donetsk]
4,284,[donetsk]
...,...,...
3613,2,"[belgorod, bryansk, dnipropetrovsk, donetsk]"
3614,2,"[belgorod, bryansk, dnipropetrovsk, donetsk]"
3615,2,"[belgorod, bryansk, dnipropetrovsk, donetsk]"
3616,2,"[belgorod, bryansk, dnipropetrovsk, donetsk]"


In [38]:
df_matched_patterns["Matched_Oblasts"].value_counts()

Matched_Oblasts
[belgorod]                         22
[]                                 21
[bryansk]                          21
[dnipropetrovsk]                   20
[belgorod, bryansk]                19
                                   ..
[irkutsk, kherson, smolensk]        1
[irkutsk, kherson, samara]          1
[irkutsk, kherson, rostov]          1
[irkutsk, kherson, odesa]           1
[dnipropetrovsk, kherson, tver]     1
Name: count, Length: 1231, dtype: int64